In [ ]:
import sys
from pathlib import Path

# Ensure project root is on sys.path regardless of where this notebook is run from
project_root = Path.cwd()
while not (project_root / 'pyproject.toml').exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print('Project root:', project_root)

In [ ]:
import torch

from src.data_prep import prepare_uji_data
from src.models import CNNJointModel
from src.training import TrainConfig, train_from_tensors

In [ ]:
# Load data (only needs to run once)
bundle = prepare_uji_data()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cfg = TrainConfig(
    lr=2e-3,
    weight_decay=1e-4,
    batch_size=256,
    val_batch_size=512,
    max_epochs=100,
    patience=10,
    print_every=5,
)

print("device:", device)
print("X train/val:", bundle.X_train.shape, bundle.X_val.shape)

In [ ]:
# Joint classification (building + floor -> 13 classes)
y_train, y_val = bundle.get_targets(["joint"])

model = CNNJointModel(in_dim=bundle.X_train.shape[1])  # in_dim=1040
print(model)
print("Parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
result = train_from_tensors(
    model=model,
    X_train=bundle.X_train,
    y_train=y_train,
    X_val=bundle.X_val,
    y_val=y_val,
    device=device,
    cfg=cfg,
)

print("\nBest epoch :", result.best_epoch)
print("Best metrics:", result.best_metrics)

In [ ]:
# Quick comparison against MLP and Transformer baselines
best = result.best_metrics
print(f"joint_accuracy    : {best['joint_accuracy']:.4f}  (transformer_joint: 0.9352)")
print(f"building_accuracy : {best['building_accuracy']:.4f}  (transformer_joint: 0.9955)")
print(f"floor_accuracy    : {best['floor_accuracy']:.4f}  (transformer_joint: 0.9352)")